# 4/4: Brain visualization
By Niloufar Shahdoust (niloufar.shahdoust@utah.edu)


                * Top view : 'axial_0', 'top'
                * Bottom view : 'axial_1', 'bottom'
                * Left : 'sagittal_0', 'left' view from left
                * Right : 'sagittal_1', 'right' view from right
                * Front : 'coronal_0', 'front'
                * Back : 'coronal_1', 'back'
                * Side front-left : 'side-fl'
                * Side front-right : 'side-fr'
                * Side back-left : 'side-bl'
                * Side back-right : 'side-br'


                

# what we do is: sagittal_0 and coronal_1

In [2]:
import os
import mat73
import numpy as np
import pandas as pd
import random
from matplotlib import cm
from matplotlib import colormaps
from ast import literal_eval
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from visbrain.objects import BrainObj, SceneObj, SourceObj
from matplotlib.colors import to_hex
from PIL import Image
from sklearn.cluster import KMeans
import matplotlib.colors as mcolors
import seaborn as sns
from fpdf import FPDF
from itertools import cycle
from matplotlib.colors import to_rgba


# ============================================================
# Required functions
# ============================================================

def get_region_color(region, region_colors):

    if region not in region_colors:
        raise ValueError(
            f"Region '{region}' not found in colormap."
        )

    return region_colors[region]


def prepare_visualization_data(
    df,
    hemisphere_selected,
    region_colors
):
    """
    Extract electrode data from both hemispheres and project all
    coordinates onto the selected hemisphere.

    For the left hemisphere:
        x = -abs(x)

    For the right hemisphere:
        x = abs(x)

    The y and z coordinates remain unchanged.
    """

    # Include electrodes from both hemispheres
    filtered_df = df[
        (df['left'] == 1) | (df['right'] == 1)
    ].copy()

    # Extract coordinates
    coordinates = filtered_df[
        [
            'coordinate_x',
            'coordinate_y',
            'coordinate_z'
        ]
    ].to_numpy(dtype=float)

    # Project both hemispheres onto the selected hemisphere
    if hemisphere_selected == 'left':

        # Mirror right-hemisphere coordinates onto the left
        coordinates[:, 0] = -np.abs(coordinates[:, 0])

    elif hemisphere_selected == 'right':

        # Mirror left-hemisphere coordinates onto the right
        coordinates[:, 0] = np.abs(coordinates[:, 0])

    else:

        raise ValueError(
            "hemisphere_selected must be 'left' or 'right'"
        )

    # Assign colors according to brain region
    colors = [
        to_rgba(
            get_region_color(region, region_colors)
        )
        for region in filtered_df['area']
    ]

    return coordinates, np.array(colors)


def plot_visbrain(
    hemisphere_selected,
    df_list,
    region_colors,
    brain_view,
    output_filename,
    elec_size=10,
    brain_zoom=1.2,
    save=False,
    output_folder='output',
    format='pdf'
):
    """
    Visualize electrode data from both hemispheres projected onto
    one selected hemisphere.

    Parameters
    ----------
    brain_zoom : float
        Controls the apparent size of the brain in the scene.
        Increase this value to make the brain larger.
    """

    os.makedirs(
        output_folder,
        exist_ok=True
    )

    scene_obj = SceneObj(
        bgcolor='white',
        size=(10000, 10000)
    )

    # Display only the selected hemisphere
    brain_obj = BrainObj(
        'B2',
        translucent=True,
        hemisphere=hemisphere_selected
    )

    # Add the brain and set its camera, view, and zoom
    scene_obj.add_to_subplot(
        brain_obj,
        row=0,
        col=0,
        use_this_cam=True,
        rotate=brain_view,
        zoom=brain_zoom
    )

    # Add electrode data from every patient
    for patient_index, df in enumerate(df_list):

        coordinates, colors = prepare_visualization_data(
            df,
            hemisphere_selected,
            region_colors
        )

        # Keep coordinates on the selected hemisphere
        if hemisphere_selected == 'left':

            mask = coordinates[:, 0] <= 0

        else:

            mask = coordinates[:, 0] >= 0

        xyz = coordinates[mask]
        filtered_colors = colors[mask]

        if len(xyz) > 0:

            iEEG_obj = SourceObj(
                name=f'iEEG_{hemisphere_selected}_{patient_index}',
                xyz=xyz,
                color=filtered_colors,
                radius_min=elec_size,
                edge_color=None
            )

            # Add electrodes to the same subplot and camera
            scene_obj.add_to_subplot(
                iEEG_obj,
                row=0,
                col=0
            )

    if save:

        screenshot_path = os.path.join(
            output_folder,
            output_filename
        )

        try:

            img_array = scene_obj.render()

            # Convert rendered array to a PIL image
            img = Image.fromarray(img_array)

            # Save through Matplotlib
            fig, ax = plt.subplots(
                figsize=(20, 20),
                dpi=1200
            )

            ax.imshow(img)
            ax.axis('off')

            plt.tight_layout()

            plt.savefig(
                screenshot_path,
                bbox_inches='tight',
                pad_inches=0,
                format=format
            )

            plt.close(fig)

            print(
                f"3D Brain Model saved at: "
                f"{screenshot_path}"
            )

        except Exception as e:

            print(
                f"Error saving the screenshot: {e}"
            )

    else:

        scene_obj.preview()


# ============================================================
# Reading data
# ============================================================

input_folder = '3_brain_visualization_preProcessing_2'
output_folder = '4_brain_visualization_main'

# Create output folder if it does not exist
os.makedirs(
    output_folder,
    exist_ok=True
)

# Find CSV files
csv_files = [
    file_name
    for file_name in os.listdir(input_folder)
    if file_name.endswith('.csv')
]

# Sort files to maintain a consistent patient order
csv_files.sort()


# ============================================================
# Load patient data
# ============================================================

df_patients = []

for file_name in csv_files:

    file_path = os.path.join(
        input_folder,
        file_name
    )

    df = pd.read_csv(file_path)

    df_patients.append(df)


# ============================================================
# Gather unique brain areas
# ============================================================

area_per_patient = [
    df['area'].dropna().unique()
    for df in df_patients
]

area_all_patients = pd.unique(
    np.concatenate(area_per_patient)
).tolist()


# ============================================================
# Create the region colors
# ============================================================

num_colors = len(area_all_patients)

colormap = plt.get_cmap('tab20')

colors_cycle = cycle(
    colormap(
        np.linspace(
            0,
            1,
            max(num_colors, 1)
        )
    )
)

region_colors = {
    region: mcolors.to_hex(next(colors_cycle))
    for region in area_all_patients
}


# ============================================================
# Sagittal view
# Both hemispheres projected onto the left hemisphere
# ============================================================

plot_visbrain(
    hemisphere_selected='left',
    df_list=df_patients,
    region_colors=region_colors,
    brain_view='sagittal_0',
    output_filename='left_sagittal.pdf',
    elec_size=30,
    brain_zoom=1.2,
    save=True,
    output_folder=output_folder,
    format='pdf'
)


# ============================================================
# Coronal view
# Both hemispheres projected onto the left hemisphere
# ============================================================

plot_visbrain(
    hemisphere_selected='left',
    df_list=df_patients,
    region_colors=region_colors,
    brain_view='coronal_1',
    output_filename='left_coronal.pdf',
    elec_size=30,
    brain_zoom=1.2,
    save=True,
    output_folder=output_folder,
    format='pdf'
)


# ============================================================
# Create the region-color legend PDF
# ============================================================

pdf = FPDF()

pdf.set_auto_page_break(
    auto=True,
    margin=15
)

pdf.add_page()

pdf.set_font(
    "Arial",
    size=12
)

# Add each region using the same color as its electrodes
for region, color in region_colors.items():

    # Convert HEX color to RGB
    rgb = tuple(
        int(color[i:i + 2], 16)
        for i in (1, 3, 5)
    )

    pdf.set_text_color(*rgb)

    pdf.cell(
        0,
        10,
        str(region),
        ln=True
    )


legend_filename = 'regions.pdf'

complete_output_path = os.path.join(
    output_folder,
    legend_filename
)

pdf.output(complete_output_path)

print(
    f"Region color PDF saved at: "
    f"{complete_output_path}"
)

complete_output_path

Creation of a scene
BrainObj(name='B2') created
    BrainObj(name='B2') added to the scene
SourceObj(name='iEEG_left_0') created
    77 sources detected
    SourceObj(name='iEEG_left_0') added to the scene
SourceObj(name='iEEG_left_1') created
    76 sources detected
    SourceObj(name='iEEG_left_1') added to the scene
SourceObj(name='iEEG_left_2') created
    59 sources detected
    SourceObj(name='iEEG_left_2') added to the scene
SourceObj(name='iEEG_left_3') created
    90 sources detected
    SourceObj(name='iEEG_left_3') added to the scene
SourceObj(name='iEEG_left_4') created
    36 sources detected
    SourceObj(name='iEEG_left_4') added to the scene
SourceObj(name='iEEG_left_5') created
    91 sources detected
    SourceObj(name='iEEG_left_5') added to the scene
SourceObj(name='iEEG_left_6') created
    51 sources detected
    SourceObj(name='iEEG_left_6') added to the scene
SourceObj(name='iEEG_left_7') created
    60 sources detected
    SourceObj(name='iEEG_left_7') added to

3D Brain Model saved at: 4_brain_visualization_main\left_sagittal.pdf


SourceObj(name='iEEG_left_2') created
    59 sources detected
    SourceObj(name='iEEG_left_2') added to the scene
SourceObj(name='iEEG_left_3') created
    90 sources detected
    SourceObj(name='iEEG_left_3') added to the scene
SourceObj(name='iEEG_left_4') created
    36 sources detected
    SourceObj(name='iEEG_left_4') added to the scene
SourceObj(name='iEEG_left_5') created
    91 sources detected
    SourceObj(name='iEEG_left_5') added to the scene
SourceObj(name='iEEG_left_6') created
    51 sources detected
    SourceObj(name='iEEG_left_6') added to the scene
SourceObj(name='iEEG_left_7') created
    60 sources detected
    SourceObj(name='iEEG_left_7') added to the scene
SourceObj(name='iEEG_left_8') created
    67 sources detected
    SourceObj(name='iEEG_left_8') added to the scene
SourceObj(name='iEEG_left_9') created
    76 sources detected
    SourceObj(name='iEEG_left_9') added to the scene
SourceObj(name='iEEG_left_10') created
    63 sources detected
    SourceObj(nam

3D Brain Model saved at: 4_brain_visualization_main\left_coronal.pdf
Region color PDF saved at: 4_brain_visualization_main\regions.pdf


'4_brain_visualization_main\\regions.pdf'